# rooms-of-kv: reproduction notebook

Companion notebook for *Two architectures that didn't help small-model agent memory on a free T4* (Medium).

Runs three retrieval architectures on the same 50-question stratified LongMemEval_S sample with Qwen3-4B-Instruct-2507, judged by Gemma 3 12B:

- **B3 Flat RAG** — top-5 by cosine similarity
- **B5 Scoped Rooms** — k-means × 16 rooms, route to top-2, retrieve top-5 within
- **B7 LLM Rerank** — retrieve top-20 flat, Gemma reranks to top-5

**Requirements:**
- Free Colab T4 (Runtime → Change runtime type → T4 GPU)
- `GOOGLE_API_KEY` Colab secret (get one at https://aistudio.google.com/apikey)
- `HF_TOKEN` Colab secret (get one at https://huggingface.co/settings/tokens)

**Total wall time:** ~2 hours (40 min per architecture + judge API time)

Results are streamed to JSONL files so you can safely kill and resume the notebook at any point.


## 1. Setup

Mount Drive, install dependencies, check GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs("/content/drive/MyDrive/hf_cache", exist_ok=True)

import torch, gc
gc.collect(); torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (no GPU!)'}")

In [ ]:
!pip install -q -U "transformers>=5.5" accelerate chromadb sentence-transformers google-genai scikit-learn

## 2. Data, tokenizer, and sample manifest

Downloads LongMemEval_S (~274 MB) to Drive on first run; cached thereafter.

The sample manifest uses seed=42 with a fixed stratification across the six question types, so every run of this notebook evaluates the same 50 questions.

In [ ]:
from huggingface_hub import hf_hub_download
import json

DATASET_PATH = '/content/drive/MyDrive/longmemeval/longmemeval_s_cleaned.json'
if not os.path.exists(DATASET_PATH):
    hf_hub_download(
        repo_id="xiaowu0162/longmemeval-cleaned",
        filename="longmemeval_s_cleaned.json",
        repo_type="dataset",
        local_dir="/content/drive/MyDrive/longmemeval",
    )
with open(DATASET_PATH) as f:
    data = json.load(f)
print(f"✓ {len(data)} questions loaded")

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
print(f"✓ Tokenizer loaded")

def session_to_text(session, session_date=None, session_id=None):
    header = f"[Session {session_id} on {session_date}]\n" if session_id else ""
    body = "\n".join(f"{turn['role'].upper()}: {turn['content']}" for turn in session)
    return header + body

def haystack_to_chunks(question, target_chunk_tokens=512):
    chunks = []
    for sid, sdate, session in zip(question["haystack_session_ids"],
                                    question["haystack_dates"],
                                    question["haystack_sessions"]):
        text = session_to_text(session, sdate, sid)
        n_tok = len(tok.encode(text, add_special_tokens=False))
        if n_tok <= target_chunk_tokens:
            chunks.append({"chunk_id": f"{sid}_0", "session_id": sid,
                          "session_date": sdate, "text": text, "n_tokens": n_tok})
        else:
            current_text, current_tok, sub_idx = "", 0, 0
            for turn in session:
                turn_text = f"{turn['role'].upper()}: {turn['content']}\n"
                turn_tok = len(tok.encode(turn_text, add_special_tokens=False))
                if current_tok + turn_tok > target_chunk_tokens and current_text:
                    chunks.append({"chunk_id": f"{sid}_{sub_idx}", "session_id": sid,
                                  "session_date": sdate,
                                  "text": f"[Session {sid} on {sdate}, part {sub_idx}]\n" + current_text,
                                  "n_tokens": current_tok})
                    sub_idx += 1
                    current_text, current_tok = turn_text, turn_tok
                else:
                    current_text += turn_text
                    current_tok += turn_tok
            if current_text:
                chunks.append({"chunk_id": f"{sid}_{sub_idx}", "session_id": sid,
                              "session_date": sdate,
                              "text": f"[Session {sid} on {sdate}, part {sub_idx}]\n" + current_text,
                              "n_tokens": current_tok})
    return chunks

def find_gold_chunks(question, chunks):
    gold_sessions = set(question["answer_session_ids"])
    return [i for i, c in enumerate(chunks) if c["session_id"] in gold_sessions]

print("✓ Chunker ready")

In [ ]:
from collections import defaultdict
import random
random.seed(42)

by_type = defaultdict(list)
for i, q in enumerate(data):
    by_type[q["question_type"]].append(i)

# Stratified sample, proportional-ish across the 6 types (50 total)
target = {
    "single-session-user": 10,
    "single-session-assistant": 8,
    "single-session-preference": 5,
    "multi-session": 12,
    "knowledge-update": 8,
    "temporal-reasoning": 7,
}
assert sum(target.values()) == 50

sample_idxs = []
for qtype, n in target.items():
    sample_idxs.extend(random.sample(by_type[qtype], min(n, len(by_type[qtype]))))

print(f"Sample size: {len(sample_idxs)}")
for qtype, n in target.items():
    print(f"  {qtype:28s} n={n}")

with open('/content/drive/MyDrive/rooms_of_kv_sample_manifest.json', 'w') as f:
    json.dump({"seed": 42, "N": len(sample_idxs), "stratification": target,
               "question_indices": sample_idxs}, f, indent=2)

## 3. Load models and judge client

Qwen3-4B (FP16, ~8 GB VRAM), MiniLM-L6-v2 embedder, and the Gemma 3 12B judge client.

**Note on the judge:** Gemma 3 12B via Google AI Studio at free tier. The notebook paces API calls below the published rate limits at time of writing.

In [ ]:
from transformers import AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from google import genai
from google.genai import types
import time, re

print("Loading Qwen3-4B...")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    dtype=torch.float16, device_map="auto", attn_implementation="sdpa",
)
devices = {p.device for p in model.parameters()}
assert devices == {torch.device('cuda:0')}, f"Model split: {devices}"
print(f"  ✓ Qwen3 VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
print(f"  ✓ Embedder loaded, dim={embedder.get_embedding_dimension()}")

gem_client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
print(f"  ✓ Gemma client ready")

In [ ]:
JUDGE_SLEEP = 2.2  # stay under free-tier RPM on Gemma 3 12B
RERANK_SLEEP = 2.2

def judge_gemma(question, gold, predicted, max_retries=3):
    """Gemma 3 12B judge. Returns (1, raw) for CORRECT, (0, raw) for INCORRECT, (None, raw) on parse failure."""
    prompt = (
        "You evaluate whether a predicted answer correctly answers a question, given the gold reference answer. "
        "Be lenient on phrasing, strict on facts.\n\n"
        "Rules:\n"
        "- CORRECT if the prediction conveys the same factual information as the gold.\n"
        "- INCORRECT if the prediction states a different number, date, duration, or named entity than the gold.\n"
        "- INCORRECT if the prediction answers a different aspect of the question (e.g., listing items instead of counting them).\n"
        "- INCORRECT if the prediction abstains ('I don't know') when the gold has a factual answer.\n"
        "- For preferences, CORRECT if the prediction aligns with the preference described in the gold.\n"
        "- Output exactly one word: CORRECT or INCORRECT.\n\n"
        f"Question: {question}\nGold: {gold}\nPrediction: {predicted}\n\nVerdict:"
    )
    for attempt in range(max_retries):
        try:
            resp = gem_client.models.generate_content(
                model="gemma-3-12b-it", contents=prompt,
                config=types.GenerateContentConfig(temperature=0, max_output_tokens=10),
            )
            verdict = (resp.text or "").strip().upper()
            if "INCORRECT" in verdict: return 0, verdict
            if "CORRECT" in verdict: return 1, verdict
            return None, verdict
        except Exception as e:
            if attempt == max_retries - 1:
                return None, f"ERR: {str(e)[:100]}"
            time.sleep(5 * (attempt + 1))

print("✓ Judge ready")

## 4. B3 Flat RAG — baseline

Standard top-5 cosine retrieval. Per-question index built in-memory (~2s), then Qwen3 generates from concatenated top-5 chunks.

**~40 min wall time on T4.** Results stream to JSONL; if interrupted, re-run to resume.

In [ ]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

def build_index_for_question(q, collection_name="haystack"):
    try: chroma_client.delete_collection(collection_name)
    except: pass
    coll = chroma_client.create_collection(collection_name, metadata={"hnsw:space": "cosine"})
    chunks = haystack_to_chunks(q)
    texts = [c["text"] for c in chunks]
    embs = embedder.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=False)
    coll.add(
        ids=[str(i) for i in range(len(chunks))],
        embeddings=embs.tolist(),
        documents=texts,
        metadatas=[{"session_id": c["session_id"]} for c in chunks],
    )
    return coll, chunks

def retrieve(coll, query, k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True)[0]
    res = coll.query(query_embeddings=[q_emb.tolist()], n_results=k)
    return [int(i) for i in res["ids"][0]]

def flat_rag_answer(q, k=5, max_new_tokens=80):
    coll, chunks = build_index_for_question(q)
    retrieved_idxs = retrieve(coll, q["question"], k=k)
    retrieved_texts = [chunks[i]["text"] for i in retrieved_idxs]
    gold_idxs = set(find_gold_chunks(q, chunks))

    context = "\n\n---\n\n".join(retrieved_texts)
    messages = [{"role": "user", "content":
        f"Below are excerpts from past conversations. Based ONLY on these excerpts, "
        f"answer the question concisely (a few words or a single sentence). If the excerpts "
        f"don't contain enough information, say so.\n\n"
        f"=== Excerpts ===\n{context}\n=== End ===\n\n"
        f"Question: {q['question']}\nAnswer:"}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    n_in = inputs.input_ids.shape[1]
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    pred = tok.decode(out[0][n_in:], skip_special_tokens=True).strip()
    return {
        "pred": pred, "retrieved_idxs": retrieved_idxs,
        "n_gold": len(gold_idxs),
        "hit_at_5": int(bool(gold_idxs & set(retrieved_idxs))),
        "recall_at_5": (len(gold_idxs & set(retrieved_idxs)) / len(gold_idxs)) if gold_idxs else 0,
        "n_chunks_total": len(chunks), "n_input_tokens": n_in,
        "peak_vram_gb": torch.cuda.max_memory_allocated()/1e9,
        "gen_latency_s": time.time() - t0,
    }

print("✓ B3 Flat RAG pipeline defined")

In [ ]:
from pathlib import Path

OUT_PATH = '/content/drive/MyDrive/rooms_of_kv_flat_rag_n50.jsonl'

done_qis = set()
if Path(OUT_PATH).exists():
    with open(OUT_PATH) as f:
        for line in f:
            try: done_qis.add(json.loads(line)["qi"])
            except: pass
    print(f"Resume: {len(done_qis)} done, skipping")
else:
    print("Fresh run")

t_run_start = time.time()
completed = 0

with open(OUT_PATH, 'a') as fout:
    for n, qi in enumerate(sample_idxs):
        if qi in done_qis: continue
        q = data[qi]
        t0 = time.time()

        try:
            result = flat_rag_answer(q)
        except Exception as e:
            result = {"error_answer": str(e)[:200]}

        if "error_answer" not in result:
            v_int, v_raw = judge_gemma(q["question"], str(q["answer"]), result["pred"])
            result["judge_verdict"] = v_int
            result["judge_raw"] = v_raw
            time.sleep(JUDGE_SLEEP)
        else:
            result["judge_verdict"] = None
            result["judge_raw"] = None

        record = {"qi": qi, "question_id": q["question_id"],
                  "question_type": q["question_type"], "question": q["question"],
                  "gold": str(q["answer"]), "total_latency_s": time.time() - t0, **result}
        fout.write(json.dumps(record) + "\n")
        fout.flush(); os.fsync(fout.fileno())
        completed += 1

        elapsed = time.time() - t_run_start
        rate = completed / elapsed if elapsed > 0 else 0
        eta = (len(sample_idxs) - len(done_qis) - completed) / rate if rate > 0 else 0
        vs = "✓" if result.get("judge_verdict") == 1 else ("✗" if result.get("judge_verdict") == 0 else "?")
        print(f"[{n+1:2d}/{len(sample_idxs)}] {q['question_type'][:20]:20s} {vs} "
              f"R@5={result.get('recall_at_5', 0):.2f} "
              f"tok={result.get('n_input_tokens', 0):4d} | ETA {eta/60:.1f}min")

print(f"\n✓ B3 Flat done. Wall time: {(time.time()-t_run_start)/60:.1f} min")

## 5. B5 Scoped Rooms — MemPalace-style hierarchical retrieval

Cluster each haystack's chunks into K=16 rooms via k-means. At query time, pick top-2 rooms by centroid cosine similarity, then top-5 chunks within that ~40-chunk pool.

Rooms are cached per question so re-running is fast after the first build.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import pickle

ROOMS_CACHE = Path('/content/drive/MyDrive/rooms_of_kv_rooms_cache.pkl')
K_ROOMS = 16

def build_rooms_for_question(q, k=K_ROOMS, seed=42):
    chunks = haystack_to_chunks(q)
    texts = [c["text"] for c in chunks]
    embs = embedder.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=False)
    n_clusters = min(k, len(chunks))
    km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=3)
    labels = km.fit_predict(embs)
    room_idxs = {r: [i for i, l in enumerate(labels) if l == r] for r in range(n_clusters)}
    return {"chunks": chunks, "embeddings": embs, "labels": labels,
            "centroids": km.cluster_centers_, "room_idxs": room_idxs}

if ROOMS_CACHE.exists():
    print("Loading rooms from cache...")
    with open(ROOMS_CACHE, 'rb') as f:
        rooms_db = pickle.load(f)
    print(f"✓ Loaded {len(rooms_db)} cached rooms")
else:
    print(f"Building rooms for {len(sample_idxs)} questions (K={K_ROOMS})...")
    rooms_db = {}
    t0 = time.time()
    for n, qi in enumerate(sample_idxs):
        rooms_db[qi] = build_rooms_for_question(data[qi], k=K_ROOMS)
        if (n+1) % 10 == 0:
            print(f"  [{n+1}/{len(sample_idxs)}] {(time.time()-t0)/(n+1):.1f}s/q avg")
    print(f"✓ Built in {(time.time()-t0)/60:.1f} min")
    with open(ROOMS_CACHE, 'wb') as f:
        pickle.dump(rooms_db, f)

In [ ]:
def scoped_retrieve(rooms, query_text, k_rooms=2, k_items=5):
    q_emb = embedder.encode([query_text], convert_to_numpy=True)[0]
    centroids = rooms["centroids"]
    centroid_norms = np.linalg.norm(centroids, axis=1)
    q_norm = np.linalg.norm(q_emb)
    room_scores = (centroids @ q_emb) / (centroid_norms * q_norm + 1e-9)
    top_rooms = np.argsort(-room_scores)[:k_rooms]
    candidate_idxs = []
    for r in top_rooms:
        candidate_idxs.extend(rooms["room_idxs"][int(r)])
    if not candidate_idxs:
        return [], {"top_rooms": top_rooms.tolist(), "candidate_count": 0}
    cand_embs = rooms["embeddings"][candidate_idxs]
    cand_norms = np.linalg.norm(cand_embs, axis=1)
    chunk_scores = (cand_embs @ q_emb) / (cand_norms * q_norm + 1e-9)
    top_local = np.argsort(-chunk_scores)[:k_items]
    retrieved_idxs = [candidate_idxs[i] for i in top_local]
    return retrieved_idxs, {
        "top_rooms": top_rooms.tolist(),
        "room_scores": [float(room_scores[r]) for r in top_rooms],
        "candidate_count": len(candidate_idxs),
    }

def scoped_rag_answer(q, rooms, k_rooms=2, k_items=5, max_new_tokens=80):
    retrieved_idxs, route_info = scoped_retrieve(rooms, q["question"], k_rooms, k_items)
    chunks = rooms["chunks"]
    retrieved_texts = [chunks[i]["text"] for i in retrieved_idxs]
    gold_idxs = set(find_gold_chunks(q, chunks))
    context = "\n\n---\n\n".join(retrieved_texts)
    messages = [{"role": "user", "content":
        f"Below are excerpts from past conversations. Based ONLY on these excerpts, "
        f"answer the question concisely (a few words or a single sentence). If the excerpts "
        f"don't contain enough information, say so.\n\n"
        f"=== Excerpts ===\n{context}\n=== End ===\n\n"
        f"Question: {q['question']}\nAnswer:"}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    n_in = inputs.input_ids.shape[1]
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    pred = tok.decode(out[0][n_in:], skip_special_tokens=True).strip()
    return {
        "pred": pred, "retrieved_idxs": retrieved_idxs,
        "n_gold": len(gold_idxs),
        "hit_at_5": int(bool(gold_idxs & set(retrieved_idxs))),
        "recall_at_5": (len(gold_idxs & set(retrieved_idxs)) / len(gold_idxs)) if gold_idxs else 0,
        "top_rooms": route_info["top_rooms"],
        "candidate_pool_size": route_info["candidate_count"],
        "n_chunks_total": len(chunks), "n_input_tokens": n_in,
        "peak_vram_gb": torch.cuda.max_memory_allocated()/1e9,
        "gen_latency_s": time.time() - t0,
    }

print("✓ B5 Scoped Rooms pipeline defined")

In [ ]:
OUT_PATH = '/content/drive/MyDrive/rooms_of_kv_scoped_rooms_n50.jsonl'

done_qis = set()
if Path(OUT_PATH).exists():
    with open(OUT_PATH) as f:
        for line in f:
            try: done_qis.add(json.loads(line)["qi"])
            except: pass
    print(f"Resume: {len(done_qis)} done, skipping")
else:
    print("Fresh run")

t_run_start = time.time()
completed = 0

with open(OUT_PATH, 'a') as fout:
    for n, qi in enumerate(sample_idxs):
        if qi in done_qis: continue
        q = data[qi]
        rooms = rooms_db[qi]
        t0 = time.time()

        try:
            result = scoped_rag_answer(q, rooms, k_rooms=2, k_items=5)
        except Exception as e:
            result = {"error_answer": str(e)[:200]}

        if "error_answer" not in result:
            v_int, v_raw = judge_gemma(q["question"], str(q["answer"]), result["pred"])
            result["judge_verdict"] = v_int
            result["judge_raw"] = v_raw
            time.sleep(JUDGE_SLEEP)
        else:
            result["judge_verdict"] = None
            result["judge_raw"] = None

        record = {"qi": qi, "question_id": q["question_id"],
                  "question_type": q["question_type"], "question": q["question"],
                  "gold": str(q["answer"]), "total_latency_s": time.time() - t0, **result}
        fout.write(json.dumps(record) + "\n")
        fout.flush(); os.fsync(fout.fileno())
        completed += 1

        elapsed = time.time() - t_run_start
        rate = completed / elapsed if elapsed > 0 else 0
        eta = (len(sample_idxs) - len(done_qis) - completed) / rate if rate > 0 else 0
        vs = "✓" if result.get("judge_verdict") == 1 else ("✗" if result.get("judge_verdict") == 0 else "?")
        print(f"[{n+1:2d}/{len(sample_idxs)}] {q['question_type'][:20]:20s} {vs} "
              f"R@5={result.get('recall_at_5', 0):.2f} "
              f"rooms={result.get('top_rooms', [])} | ETA {eta/60:.1f}min")

print(f"\n✓ B5 Scoped done. Wall time: {(time.time()-t_run_start)/60:.1f} min")

## 6. B7 LLM Rerank — retrieve top-20 flat, Gemma reranks to top-5

Retrieve the top 20 chunks via flat cosine, ask Gemma 3 12B to pick the 5 most useful for answering, pass those to Qwen3.

**~45 min wall time** — the rerank API call adds ~10-30s per question. Most of the time is judge + rerank throttling.

In [ ]:
chroma2 = chromadb.Client(Settings(anonymized_telemetry=False))

def flat_retrieve_topn(q, n=20):
    try: chroma2.delete_collection("haystack_rr")
    except: pass
    coll = chroma2.create_collection("haystack_rr", metadata={"hnsw:space": "cosine"})
    chunks = haystack_to_chunks(q)
    texts = [c["text"] for c in chunks]
    embs = embedder.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=False)
    coll.add(ids=[str(i) for i in range(len(chunks))],
             embeddings=embs.tolist(), documents=texts,
             metadatas=[{"session_id": c["session_id"]} for c in chunks])
    q_emb = embedder.encode([q["question"]], convert_to_numpy=True)[0]
    res = coll.query(query_embeddings=[q_emb.tolist()], n_results=n)
    return [int(i) for i in res["ids"][0]], chunks

def rerank_with_gemma(question, candidate_chunks, keep=5, max_retries=3):
    numbered = "\n\n".join(
        f"[{i}] {c['text'][:600]}..." if len(c['text']) > 600 else f"[{i}] {c['text']}"
        for i, c in enumerate(candidate_chunks)
    )
    prompt = (
        f"You are ranking conversation excerpts by their relevance to answering a user's question. "
        f"Return ONLY the indices of the {keep} most useful excerpts, in order of usefulness, "
        f"as a comma-separated list (e.g., '3,7,1,12,0'). "
        f"An excerpt is useful if it contains information needed to answer the question — "
        f"not merely topically related content.\n\n"
        f"Question: {question}\n\nExcerpts:\n{numbered}\n\n"
        f"Top-{keep} indices (comma-separated, no explanation):"
    )
    for attempt in range(max_retries):
        try:
            resp = gem_client.models.generate_content(
                model="gemma-3-12b-it", contents=prompt,
                config=types.GenerateContentConfig(temperature=0, max_output_tokens=40),
            )
            text = (resp.text or "").strip()
            parts = text.replace("\n", ",").replace(" ", "").split(",")
            ranked = []
            for p in parts:
                p = p.strip("[]().")
                if p.isdigit():
                    idx = int(p)
                    if 0 <= idx < len(candidate_chunks) and idx not in ranked:
                        ranked.append(idx)
                if len(ranked) >= keep: break
            if ranked: return ranked, text
        except Exception as e:
            if attempt == max_retries - 1:
                return None, f"ERR: {str(e)[:100]}"
            time.sleep(5 * (attempt + 1))
    return None, text

def reranked_rag_answer(q, n_retrieve=20, n_keep=5, max_new_tokens=80):
    topN_idxs, chunks = flat_retrieve_topn(q, n=n_retrieve)
    candidates = [chunks[i] for i in topN_idxs]
    gold_idxs = set(find_gold_chunks(q, chunks))

    t_rr0 = time.time()
    kept_local, rerank_raw = rerank_with_gemma(q["question"], candidates, keep=n_keep)
    rerank_latency = time.time() - t_rr0

    if kept_local is None:
        kept_local = list(range(min(n_keep, len(candidates))))
        rerank_raw = "FALLBACK"

    retrieved_idxs = [topN_idxs[i] for i in kept_local]
    retrieved_texts = [chunks[i]["text"] for i in retrieved_idxs]

    context = "\n\n---\n\n".join(retrieved_texts)
    messages = [{"role": "user", "content":
        f"Below are excerpts from past conversations. Based ONLY on these excerpts, "
        f"answer the question concisely (a few words or a single sentence). If the excerpts "
        f"don't contain enough information, say so.\n\n"
        f"=== Excerpts ===\n{context}\n=== End ===\n\n"
        f"Question: {q['question']}\nAnswer:"}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    n_in = inputs.input_ids.shape[1]
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    pred = tok.decode(out[0][n_in:], skip_special_tokens=True).strip()

    return {
        "pred": pred, "retrieved_idxs": retrieved_idxs, "topN_idxs": topN_idxs,
        "n_gold": len(gold_idxs),
        "hit_at_5_after_rerank": int(bool(gold_idxs & set(retrieved_idxs))),
        "recall_at_5_after_rerank": (len(gold_idxs & set(retrieved_idxs)) / len(gold_idxs)) if gold_idxs else 0,
        "hit_at_20_before_rerank": int(bool(gold_idxs & set(topN_idxs))),
        "recall_at_20_before_rerank": (len(gold_idxs & set(topN_idxs)) / len(gold_idxs)) if gold_idxs else 0,
        "rerank_raw": rerank_raw, "rerank_latency_s": rerank_latency,
        "n_chunks_total": len(chunks), "n_input_tokens": n_in,
        "peak_vram_gb": torch.cuda.max_memory_allocated()/1e9,
        "gen_latency_s": time.time() - t0,
    }

print("✓ B7 Rerank pipeline defined")

In [ ]:
OUT_PATH = '/content/drive/MyDrive/rooms_of_kv_rerank_n50.jsonl'

done_qis = set()
if Path(OUT_PATH).exists():
    with open(OUT_PATH) as f:
        for line in f:
            try: done_qis.add(json.loads(line)["qi"])
            except: pass
    print(f"Resume: {len(done_qis)} done, skipping")
else:
    print("Fresh run")

t_run_start = time.time()
completed = 0

with open(OUT_PATH, 'a') as fout:
    for n, qi in enumerate(sample_idxs):
        if qi in done_qis: continue
        q = data[qi]
        t0 = time.time()

        try:
            result = reranked_rag_answer(q)
        except Exception as e:
            result = {"error_answer": str(e)[:200]}
        time.sleep(RERANK_SLEEP)

        if "error_answer" not in result:
            v_int, v_raw = judge_gemma(q["question"], str(q["answer"]), result["pred"])
            result["judge_verdict"] = v_int
            result["judge_raw"] = v_raw
            time.sleep(JUDGE_SLEEP)
        else:
            result["judge_verdict"] = None
            result["judge_raw"] = None

        record = {"qi": qi, "question_id": q["question_id"],
                  "question_type": q["question_type"], "question": q["question"],
                  "gold": str(q["answer"]), "total_latency_s": time.time() - t0, **result}
        fout.write(json.dumps(record) + "\n")
        fout.flush(); os.fsync(fout.fileno())
        completed += 1

        elapsed = time.time() - t_run_start
        rate = completed / elapsed if elapsed > 0 else 0
        eta = (len(sample_idxs) - len(done_qis) - completed) / rate if rate > 0 else 0
        vs = "✓" if result.get("judge_verdict") == 1 else ("✗" if result.get("judge_verdict") == 0 else "?")
        h20 = result.get('hit_at_20_before_rerank', 0)
        h5 = result.get('hit_at_5_after_rerank', 0)
        print(f"[{n+1:2d}/{len(sample_idxs)}] {q['question_type'][:20]:20s} {vs} "
              f"H20→5={h20}→{h5} | ETA {eta/60:.1f}min")

print(f"\n✓ B7 Rerank done. Wall time: {(time.time()-t_run_start)/60:.1f} min")

## 7. Three-way comparison

Reproduces the headline table and figure from the Medium article.

In [ ]:
from collections import defaultdict
from math import comb

def load(p):
    with open(p) as f:
        return [json.loads(l) for l in f if l.strip()]

flat = {r["qi"]: r for r in load('/content/drive/MyDrive/rooms_of_kv_flat_rag_n50.jsonl')}
scoped = {r["qi"]: r for r in load('/content/drive/MyDrive/rooms_of_kv_scoped_rooms_n50.jsonl')}
rerank = {r["qi"]: r for r in load('/content/drive/MyDrive/rooms_of_kv_rerank_n50.jsonl')}

def acc_by_type(recs):
    bt = defaultdict(list)
    for r in recs.values():
        bt[r["question_type"]].append(r)
    return {qt: (sum(1 for r in bt[qt] if r.get("judge_verdict") == 1), len(bt[qt])) for qt in bt}

types_order = ["single-session-user", "single-session-assistant", "single-session-preference",
               "multi-session", "knowledge-update", "temporal-reasoning"]

fa = acc_by_type(flat); sa = acc_by_type(scoped); ra = acc_by_type(rerank)

print(f"{'Type':<28} {'N':>3}  {'B3 Flat':>10} {'B5 Scoped':>12} {'B7 Rerank':>12}")
print("-" * 75)
for qt in types_order:
    fc, fn = fa.get(qt, (0,0))
    sc, sn = sa.get(qt, (0,0))
    rc, rn = ra.get(qt, (0,0))
    print(f"{qt:<28} {fn:>3}  {fc}/{fn} ({fc/fn*100:>4.0f}%)  {sc}/{sn} ({sc/sn*100:>4.0f}%)  {rc}/{rn} ({rc/rn*100:>4.0f}%)")

print("-" * 75)
fc_total = sum(1 for r in flat.values() if r.get("judge_verdict") == 1)
sc_total = sum(1 for r in scoped.values() if r.get("judge_verdict") == 1)
rc_total = sum(1 for r in rerank.values() if r.get("judge_verdict") == 1)
print(f"{'OVERALL':<28} {50:>3}  {fc_total}/50 ({fc_total*2:>3}%)  {sc_total}/50 ({sc_total*2:>3}%)  {rc_total}/50 ({rc_total*2:>3}%)")

# McNemar exact on flip pairs
def mcnemar(a, b):
    a_wins = b_wins = 0
    for qi in a:
        av, bv = a[qi].get("judge_verdict"), b[qi].get("judge_verdict")
        if av is None or bv is None: continue
        if av == 1 and bv == 0: a_wins += 1
        elif av == 0 and bv == 1: b_wins += 1
    total_disc = a_wins + b_wins
    if total_disc == 0: return a_wins, b_wins, 1.0
    k = min(a_wins, b_wins)
    p = 2 * sum(comb(total_disc, i) for i in range(k+1)) / (2**total_disc)
    return a_wins, b_wins, min(p, 1.0)

for name, a, b in [("flat vs scoped", flat, scoped),
                   ("flat vs rerank", flat, rerank),
                   ("scoped vs rerank", scoped, rerank)]:
    aw, bw, p = mcnemar(a, b)
    print(f"\nMcNemar {name}: {aw} vs {bw} discordant, p={p:.3f} two-sided")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

short_labels = ["ss-user\n(n=10)", "ss-assist\n(n=8)", "ss-pref\n(n=5)",
                "multi-sess\n(n=12)", "know-upd\n(n=8)", "temporal\n(n=7)"]

flat_acc = [fa[qt][0]/fa[qt][1] for qt in types_order]
scoped_acc = [sa[qt][0]/sa[qt][1] for qt in types_order]
rerank_acc = [ra[qt][0]/ra[qt][1] for qt in types_order]

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(types_order))
width = 0.27

bars = [
    ax.bar(x - width, flat_acc, width, label=f"B3 Flat RAG ({fc_total*2}%)",
           color="#4c72b0", edgecolor='white'),
    ax.bar(x, scoped_acc, width, label=f"B5 Scoped Rooms ({sc_total*2}%)",
           color="#dd8452", edgecolor='white'),
    ax.bar(x + width, rerank_acc, width, label=f"B7 Rerank ({rc_total*2}%)",
           color="#55a868", edgecolor='white'),
]
for bset in bars:
    for bar in bset:
        h = bar.get_height()
        ax.annotate(f"{h*100:.0f}%", xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

ax.axvspan(2.5, 5.5, alpha=0.06, color='red', zorder=0)
ax.text(4.0, 1.08, "hard categories (retrieval OK, reasoning fails)",
        ha='center', va='top', fontsize=9, style='italic', color='#663333')

ax.set_xlabel("Question type", fontsize=11)
ax.set_ylabel("Accuracy (correct / n, parse-fails count as wrong)", fontsize=11)
ax.set_title("LongMemEval_S accuracy by question type across three architectures\n"
             "Qwen3-4B-Instruct-2507 on free Colab T4, N=50 stratified, seed=42",
             fontsize=12, pad=12)
ax.set_xticks(x); ax.set_xticklabels(short_labels, fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_yticks(np.arange(0, 1.01, 0.1))
ax.set_yticklabels([f"{int(p*100)}%" for p in np.arange(0, 1.01, 0.1)])
ax.legend(loc="upper left", fontsize=10, framealpha=0.95)
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/rooms_of_kv_fig2.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Saved figure")

## Done

The three JSONL files in `/content/drive/MyDrive/` contain every per-question raw record used in the Medium article:

- `rooms_of_kv_flat_rag_n50.jsonl`
- `rooms_of_kv_scoped_rooms_n50.jsonl`
- `rooms_of_kv_rerank_n50.jsonl`

Figure: `rooms_of_kv_fig2.png`.

**If you want to extend this:** swap the model in Section 3 (try `Qwen/Qwen3-8B-Instruct-AWQ` for INT4 weights at ~5 GB on T4), rerun Sections 4/5/6, and compare against the flat baseline. The sample manifest (seed=42) stays constant so any comparison is apples-to-apples.
